In [ ]:
from pathlib import Path
import os
import re

OP_DIR = Path.cwd().resolve()
# 在 op/ 下运行 notebook 时，仓库根目录为上一级
REPO_ROOT = OP_DIR.parent if (OP_DIR / "rag_txt.ipynb").exists() else OP_DIR


def load_env_file(env_path: Path | None = None):
    """从仓库根目录 .env 读取 RAGFlow 配置（与 scripts/optimization 一致）。"""
    candidates = [
        env_path,
        REPO_ROOT / ".env",
        Path(".env"),
    ]
    env_file = next((p for p in candidates if p and p.is_file()), None)
    if not env_file:
        raise FileNotFoundError(f"未找到 .env（已尝试 {REPO_ROOT / '.env'}）")

    with env_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key, value)

    print(f"已加载: {env_file}")


load_env_file()

DATASET_ID = os.getenv("RAGFLOW_DATASET_ID", "").strip()
API_KEY = os.getenv("RAGFLOW_API_KEY", "").strip()
ADDRESS = os.getenv("RAGFLOW_ADDRESS", "").strip()

if not ADDRESS:
    # 与 vllm：RAGFLOW_RETRIEVAL_URL=http://host:port/api/v1/retrieval
    url = os.getenv("RAGFLOW_RETRIEVAL_URL", "").strip().strip('"').strip("'")
    m = re.match(r"https?://([^/]+)", url)
    if m:
        ADDRESS = m.group(1)

missing = [n for n, v in {
    "RAGFLOW_ADDRESS (或 RAGFLOW_RETRIEVAL_URL)": ADDRESS,
    "RAGFLOW_DATASET_ID": DATASET_ID,
    "RAGFLOW_API_KEY": API_KEY,
}.items() if not v]

if missing:
    raise RuntimeError(f".env 缺少: {', '.join(missing)}")

print(f"RAGFlow API host: {ADDRESS}")
print(f"Dataset ID: {DATASET_ID[:8]}..." if len(DATASET_ID) > 8 else DATASET_ID)

In [ ]:
import time
import json
import mimetypes
import requests
from pathlib import Path
from contextlib import ExitStack

# 若未先跑 cell 0
try:
    OP_DIR
except NameError:
    OP_DIR = Path.cwd().resolve()

# =========================
# 配置
# =========================

# 待上传的 .md：默认 op/test（试验）；可改为 op/kb_upload_staging 或具体子目录
DATA_DIR = Path(os.getenv("RAGFLOW_UPLOAD_DIR", OP_DIR / "test"))
RECURSIVE = os.getenv("RAGFLOW_UPLOAD_RECURSIVE", "").lower() in ("1", "true", "yes")

BATCH_SIZE = int(os.getenv("RAGFLOW_UPLOAD_BATCH", "10"))
SLEEP_SECONDS = float(os.getenv("RAGFLOW_UPLOAD_SLEEP", "3"))
DRY_RUN = os.getenv("RAGFLOW_UPLOAD_DRY_RUN", "").lower() in ("1", "true", "yes")

BASE_URL = f"http://{ADDRESS}/api/v1/datasets/{DATASET_ID}"
UPLOAD_URL = f"{BASE_URL}/documents"
PARSE_URL = f"{BASE_URL}/chunks"
LIST_URL = f"{BASE_URL}/documents"


headers = {
    "Authorization": f"Bearer {API_KEY}"
}


def request_json(method, url, **kwargs):
    """统一请求，并打印可诊断信息。"""
    try:
        resp = requests.request(method, url, timeout=120, **kwargs)
    except requests.RequestException as e:
        raise RuntimeError(f"HTTP 请求失败: {e}") from e

    print(f"\n[{method}] {url}")
    print(f"HTTP status: {resp.status_code}")

    try:
        data = resp.json()
        print("Response JSON:")
        print(json.dumps(data, ensure_ascii=False, indent=2)[:3000])
    except Exception:
        print("Response TEXT:")
        print(resp.text[:3000])
        raise RuntimeError("响应不是 JSON，无法判断 RAGFlow 业务状态。")

    if resp.status_code != 200:
        raise RuntimeError(f"HTTP 状态码异常: {resp.status_code}")

    if data.get("code") != 0:
        raise RuntimeError(
            f"RAGFlow 返回业务错误: code={data.get('code')}, "
            f"message={data.get('message')}"
        )

    return data


def list_local_md_files():
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            f"目录不存在: {DATA_DIR.resolve()}\n"
            "请先创建 op/test 并放入要上传的 .md，或设置环境变量 RAGFLOW_UPLOAD_DIR"
        )

    if RECURSIVE:
        files = sorted(DATA_DIR.rglob("*.md"))
    else:
        files = sorted(
            p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".md"
        )
    return files


def upload_batch(paths):
    """
    上传一批 md 文件。
    返回 RAGFlow 创建出的 document ids。
    """
    with ExitStack() as stack:
        files = []

        for path in paths:
            file_obj = stack.enter_context(open(path, "rb"))

            mime_type = mimetypes.guess_type(path.name)[0] or "text/markdown"

            # RAGFlow 文档要求字段名是 file。
            files.append(
                (
                    "file",
                    (
                        path.name,
                        file_obj,
                        mime_type,
                    ),
                )
            )

        # 注意：不要手动设置 Content-Type。
        # requests 会自动设置 multipart/form-data boundary。
        data = request_json(
            "POST",
            UPLOAD_URL,
            headers=headers,
            files=files,
            params={"type": "local"},
        )

    docs = data.get("data") or []
    document_ids = []

    for doc in docs:
        doc_id = doc.get("id")
        name = doc.get("name")
        run = doc.get("run")
        if doc_id:
            document_ids.append(doc_id)
        print(f"创建文档: name={name}, id={doc_id}, run={run}")

    if len(document_ids) != len(paths):
        print(
            f"警告：本批上传 {len(paths)} 个文件，但返回 {len(document_ids)} 个 document id。"
        )

    return document_ids


def parse_documents(document_ids):
    """
    触发 RAGFlow 解析文档。
    """
    if not document_ids:
        return

    payload = {
        "document_ids": document_ids
    }

    request_json(
        "POST",
        PARSE_URL,
        headers={**headers, "Content-Type": "application/json"},
        json=payload,
    )

    print(f"已提交解析任务: {len(document_ids)} 个文档")


def verify_documents_by_name(names):
    """
    上传后用 List documents 检查文档是否存在。
    """
    print("\n开始校验上传结果...")

    found = set()

    # RAGFlow list documents 支持分页。
    page = 1
    page_size = 100

    while True:
        data = request_json(
            "GET",
            LIST_URL,
            headers=headers,
            params={
                "page": page,
                "page_size": page_size,
                "orderby": "create_time",
                "desc": "true",
            },
        )

        payload = data.get("data") or {}

        # 不同版本可能是 data.docs / data.items / data 结构，做兼容。
        docs = (
            payload.get("docs")
            or payload.get("items")
            or payload.get("documents")
            or payload.get("data")
            or []
        )

        if not docs:
            break

        for doc in docs:
            name = doc.get("name")
            if name in names:
                found.add(name)
                print(f"已找到: {name}, id={doc.get('id')}, run={doc.get('run')}")

        if len(docs) < page_size:
            break

        page += 1

        if page > 20:
            break

    missing = set(names) - found

    print(f"\n校验结果：找到 {len(found)}/{len(names)} 个文件")

    if missing:
        print("未找到的文件：")
        for name in sorted(missing):
            print(f"- {name}")

    return found, missing


def batch_upload_and_parse():
    md_files = list_local_md_files()
    total = len(md_files)

    print(f"开始上传 {total} 个 Markdown 文件")
    print(f"数据目录: {DATA_DIR.resolve()}")
    print(f"Dataset ID: {DATASET_ID}")

    if total == 0:
        print("没有找到 .md 文件")
        return

    all_doc_ids = []
    all_names = [p.name for p in md_files]

    for start in range(0, total, BATCH_SIZE):
        batch = md_files[start:start + BATCH_SIZE]
        print("\n" + "=" * 80)
        print(f"上传批次: {start + 1} - {start + len(batch)} / {total}")
        print([p.name for p in batch])

        try:
            doc_ids = upload_batch(batch)
            all_doc_ids.extend(doc_ids)

            # 上传后立即提交解析
            parse_documents(doc_ids)

        except Exception as e:
            print(f"本批失败: {e}")

        time.sleep(SLEEP_SECONDS)

    print("\n" + "=" * 80)
    print(f"上传阶段完成，获得 document_id 数量: {len(all_doc_ids)}")

    verify_documents_by_name(all_names)

    print("\n全部流程结束。")


def list_remote_documents(max_pages: int = 3):
    """仅列出远端最近文档（不上传）。"""
    page = 1
    while page <= max_pages:
        data = request_json(
            "GET",
            LIST_URL,
            headers=headers,
            params={"page": page, "page_size": 20, "orderby": "create_time", "desc": "true"},
        )
        payload = data.get("data") or {}
        docs = payload.get("docs") or payload.get("items") or []
        if not docs:
            break
        for doc in docs:
            print(f"  {doc.get('name')} | id={doc.get('id')} | run={doc.get('run')}")
        page += 1


if __name__ == "__main__":
    print("远端 dataset 最近文档（预览）:")
    try:
        list_remote_documents(max_pages=1)
    except Exception as e:
        print(f"列表失败（检查 RAGFLOW_ADDRESS 与服务）: {e}")

    if DRY_RUN:
        md_files = list_local_md_files()
        print(f"\n[DRY_RUN] 将上传 {len(md_files)} 个文件，目录={DATA_DIR.resolve()}")
        for p in md_files[:20]:
            print(" ", p.name)
        if len(md_files) > 20:
            print(f"  ... 共 {len(md_files)} 个")
    else:
        batch_upload_and_parse()

## 使用说明

1. **先运行 cell 0**（读仓库根目录 `.env`）。
2. **与远端对齐（推荐）**：`python op/rag_kb_sync.py sync --dry-run` 查看需更新的文件；确认后 `python op/rag_kb_sync.py sync`（按 SHA256 删远端旧文档/重复 `(1)` → 上传 `op/` 最新版 → 触发 `/chunks` 向量化）。
3. **只读对比**：cell 4 或 `python op/rag_kb_sync.py diff --tier l2`。
4. **快照下载**：cell 5 或 `python op/rag_kb_sync.py download` → `op/ragflow_export/`（已 gitignore）。
5. **试验上传**：`op/test/` + cell 3；`export RAGFLOW_UPLOAD_DRY_RUN=1` 后 cell 1 仅预览。

`sync` 会跳过与远端内容一致的 500+ API 页，仅处理 **缺失 / 内容变更 / 重复 basename** 的文件。

In [ ]:
# 生成 op/test 试验上传包（L2 已改样例 + ingest 说明）
from pathlib import Path
import shutil

try:
    OP_DIR
except NameError:
    OP_DIR = Path.cwd().resolve()

STAGING = OP_DIR / "test"
SOURCES = [
    OP_DIR / "RAG_qiskit_human_eval_runtime" / "heval_runtime_sampler_estimator.md",
    OP_DIR / "RAG_qiskit_recipes" / "recipe_statevector_quantum_info_bell.md",
    OP_DIR / "RAG_qiskit_api" / "HUMANEVAL_RAG_INGEST_NOTE.md",
    OP_DIR / "RAG_KB_CONTENT_SCOPE.md",
]

STAGING.mkdir(parents=True, exist_ok=True)
for src in SOURCES:
    if src.is_file():
        shutil.copy2(src, STAGING / src.name)
        print("copied", src.name)
    else:
        print("skip missing", src)

print("staging:", STAGING.resolve(), "files:", len(list(STAGING.glob("*.md"))))

In [ ]:
# 只读：对比 op/ 与 RAGFlow dataset（不上传、不删除）
import subprocess
import sys

REPO = OP_DIR.parent
script = OP_DIR / "rag_kb_sync.py"
for tier in ("l2", "all"):
    print("=" * 60, f"diff --tier {tier}", "=" * 60)
    subprocess.run([sys.executable, str(script), "diff", "--tier", tier], cwd=REPO, check=False)

In [ ]:
# 从 RAGFlow 下载当前 dataset 全部文档 -> op/ragflow_export/
# API: GET /api/v1/datasets/{id}/documents/{document_id} 返回原始文件字节
import subprocess
import sys

out = OP_DIR / "ragflow_export"
subprocess.run(
    [sys.executable, str(OP_DIR / "rag_kb_sync.py"), "download", "--out", str(out)],
    cwd=OP_DIR.parent,
    check=True,
)
print("see", out / "index.json", "and", out / "files")

In [ ]:
# 删除 deprecated 旧 runtime 文档（op/ + RAGFlow），并 sync L2 canonical
import subprocess
import sys

script = OP_DIR / "rag_kb_sync.py"
# subprocess.run([sys.executable, str(script), "retire-deprecated", "--dry-run"], cwd=OP_DIR.parent, check=True)
subprocess.run([sys.executable, str(script), "retire-deprecated"], cwd=OP_DIR.parent, check=True)